# Tier 1 — Grid+CV Transformers (Kaggle GPU)

Mesmo protocolo do Colab (`tier1_gridcv_transformers_colab.ipynb`), adaptado para Kaggle.

**Antes de rodar:** Settings → Accelerator → GPU T4 x2 (ou P100).

**Resume:** Se a sessão cair, faça download de `tier1_gridcv_transformers.json` em
Output, suba como dataset Kaggle e ajuste `RESUME_DATASET` abaixo.

In [ ]:
# ── Célula 1: Verifica GPU ──────────────────────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('GPU não detectada — ative em Settings > Accelerator')

In [ ]:
# ── Célula 2: Clonar repo ───────────────────────────────────────────────────
import os, subprocess

GIT_URL = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'

if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase'], check=True)
else:
    subprocess.run(['git', 'clone', GIT_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
!git log --oneline -3
print(f'Dir: {os.getcwd()}')

In [ ]:
# ── Célula 3: Dependências ──────────────────────────────────────────────────
!pip install -q entmax einops scikit-posthocs
# torch, sklearn, numpy, scipy, pandas já presentes no Kaggle
import torch, sklearn, numpy
print(f'torch {torch.__version__} | sklearn {sklearn.__version__} | numpy {numpy.__version__}')

In [ ]:
# ── Célula 4: Datasets Tier 1 ───────────────────────────────────────────────
!python scripts/download_data.py --tier 1
!ls -lh data/raw/

In [ ]:
# ── Célula 5: Resume de sessão anterior (opcional) ──────────────────────────
# Se quiser continuar de onde parou:
# 1. Suba o JSON anterior como Kaggle dataset
# 2. Ajuste o caminho abaixo e descomente

import shutil
from pathlib import Path

OUTPUT_FILE = Path('results/tier1_gridcv_transformers.json')
OUTPUT_FILE.parent.mkdir(exist_ok=True)

# RESUME_PATH = Path('/kaggle/input/SEU-DATASET/tier1_gridcv_transformers.json')
# if RESUME_PATH.exists():
#     shutil.copy(RESUME_PATH, OUTPUT_FILE)
#     import json; n = len(json.load(open(OUTPUT_FILE)))
#     print(f'Restaurado: {n} entries')

if OUTPUT_FILE.exists():
    import json
    n = len(json.load(open(OUTPUT_FILE)))
    print(f'Iniciando com {n} entries já presentes')
else:
    print('Começando do zero')

In [ ]:
# ── Célula 6: Inspecionar grade ─────────────────────────────────────────────
import sys; sys.path.insert(0, '.')
from src.tuning.grids import GRIDS, grid_size

TRANSFORMER_MODELS = [
    'FTTransformer_softmax', 'FTTransformer_topk',
    'FTTransformer_entmax', 'FTTransformer_sparsemax',
    'SAINTColnorm',
    'FTTransformerCURColnorm',
]

print(f'{"Modelo":<28}{"Grid":>6}{"Fits/entry":>12}')
print('-'*48)
total_fits = 0
for m in TRANSFORMER_MODELS:
    g = grid_size(m)
    fits = g * 5
    total_fits += fits * 9 * 30
    print(f'{m:<28}{g:>6}{fits:>12}')
print(f'\nTotal fits: {total_fits:,} (~{total_fits*2/3600:.0f}h estimado em T4)')

In [ ]:
# ── Célula 7: Rodar ─────────────────────────────────────────────────────────
models_str = ' '.join(TRANSFORMER_MODELS)
datasets_str = 'AI4I'  # só o novo dataset — os outros 9 já estão completos (1620/1620)

!python -u scripts/run_tier1_gridcv.py \
    --models {models_str} \
    --datasets {datasets_str} \
    --output results/tier1_gridcv_transformers.json \
    2>&1 | tee /kaggle/working/run.log

In [ ]:
# ── Célula 8: Resumo final ──────────────────────────────────────────────────
import json, statistics as st
from collections import defaultdict

records = json.load(open('results/tier1_gridcv_transformers.json'))
ok = [r for r in records if r.get('status') == 'ok']
n_expected = len(TRANSFORMER_MODELS) * len(datasets_str.split()) * 30
print(f'Completos: {len(ok)}/{n_expected} ({len(ok)/n_expected*100:.0f}%)')

f1 = defaultdict(list)
for r in ok:
    f1[r['variant']].append(r['test_f1_macro'])

print(f'\n{"Modelo":<30}  F1-macro  n')
for v, vals in sorted(f1.items(), key=lambda x: -st.mean(x[1])):
    print(f'{v:<30}  {st.mean(vals):.4f}  {len(vals)}')

# Copiar para output raiz do Kaggle (aparece em Output para download)
import shutil
shutil.copy('results/tier1_gridcv_transformers.json', '/kaggle/working/tier1_gridcv_transformers.json')
print('\nArquivo disponível em Output para download.')